In [ ]:
import torch
from ellzaf_ml.models import GhostFaceNetsV2

IMAGE_SIZE = 256

#return classification
model = GhostFaceNetsV2(image_size=IMAGE_SIZE, num_classes=0, width=1, dropout=0.)
img = torch.randn(3, 3, IMAGE_SIZE, IMAGE_SIZE)
#forward pass
model.eval()
with torch.no_grad():
    logits = model(img)
    print(logits.shape)  # should be [3, 0]
#return embedding
model = GhostFaceNetsV2(image_size=IMAGE_SIZE, num_classes=0, width=1, dropout=0., return_embedding=True)
img = torch.randn(3, 3, IMAGE_SIZE, IMAGE_SIZE)
#forward pass

            
emb = model(img)

In [22]:
from collections import defaultdict
import os
import numpy as np
import cv2
import torch
from ellzaf_ml.models import GhostFaceNetsV2

# Constants
IMAGE_SIZE = 256

# Initialize the model
model = GhostFaceNetsV2(image_size=IMAGE_SIZE, num_classes=0, width=1, dropout=0.)

# Set the model to evaluation mode - THIS IS THE KEY FIX
model.eval()

# Specify your database path
database_path = "/home/hbvision/mirsaid/smart-office/data/ilhan-aligned"
database_dict = {}  # Using a regular dict is fine here

# Process each image in the database
for file_name in os.listdir(database_path):
    if not file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue
        
    img_full_path = os.path.join(database_path, file_name)
    
    # Load the image and preprocess
    img = cv2.imread(img_full_path)
    if img is None:
        print(f"Could not read image: {img_full_path}")
        continue
        
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Convert to tensor and add batch dimension
    img = torch.from_numpy(img).float()
    img = img.unsqueeze(0)  # Add batch dimension
    img = img.permute(0, 3, 1, 2)  # Change from HWC to CHW
    img = img / 255.0  # Normalize to [0, 1]
    
    # Compute embeddings with no_grad to save memory and computation
    with torch.no_grad():
        encodings = model(img)
    
    # Store the embeddings
    person_id = file_name.split(".")[0]
    database_dict[person_id] = encodings.cpu().numpy()
    
    print(f"Processed {file_name}, embedding shape: {encodings.shape}")

# Save the database dictionary for later use
import pickle
with open("face_embeddings_database.pkl", "wb") as f:
    pickle.dump(database_dict, f)

print(f"Successfully processed {len(database_dict)} faces and saved embeddings.")

Processed RobiyaAbdullayeva.jpg, embedding shape: torch.Size([1, 512])
Processed Oydina Yo'ldasheva.jpg, embedding shape: torch.Size([1, 512])
Processed Mohira Murodova.jpg, embedding shape: torch.Size([1, 512])
Processed Akmal Nazarov.jpg, embedding shape: torch.Size([1, 512])
Processed Sadoqat Murodaliyeva.jpg, embedding shape: torch.Size([1, 512])
Processed Zohida Madraximova.jpg, embedding shape: torch.Size([1, 512])
Processed Yoqubjon Yusufxanov.jpg, embedding shape: torch.Size([1, 512])
Processed NasibaJo'rabayeva.jpg, embedding shape: torch.Size([1, 512])
Processed Maftuna Mahmuda.jpg, embedding shape: torch.Size([1, 512])
Processed ZuhraUlkanova.jpg, embedding shape: torch.Size([1, 512])
Processed Setora.jpg, embedding shape: torch.Size([1, 512])
Processed ShirmonoyMuhammadaminova.jpg, embedding shape: torch.Size([1, 512])
Processed DilshodaJalilova.jpg, embedding shape: torch.Size([1, 512])
Processed Marg'uba Ro'zmaliyeva.jpg, embedding shape: torch.Size([1, 512])
Processed No

In [24]:
database_dict.keys()

dict_keys(['RobiyaAbdullayeva', "Oydina Yo'ldasheva", 'Mohira Murodova', 'Akmal Nazarov', 'Sadoqat Murodaliyeva', 'Zohida Madraximova', 'Yoqubjon Yusufxanov', "NasibaJo'rabayeva", 'Maftuna Mahmuda', 'ZuhraUlkanova', 'Setora', 'ShirmonoyMuhammadaminova', 'DilshodaJalilova', "Marg'uba Ro'zmaliyeva", 'NozimaShokirova', "NafisaTurg'unova", 'Barno Shamsiddinova', 'MohidilOdiljonova', 'FarzonaSalohiddinova', 'Azizbek Dedaxanov', 'FeruzaAtamkulova', "Ulug'bekErkinov", 'Nodira Umarova', 'Feruza Yusupova', 'Gulmira Rustamova', "Turg'unoy Razzoqova", 'Yuldosheva Naima', 'Qodirxon Qosimov', 'Oydina Sharipova', "Dilafruz Yo'ldoshbayeva", 'Zamira Nurmatova', 'UmidaBozorova', 'FotimaUlkanova', 'Zahro Uroqova', 'Mashrab Muydinov', 'Abdukarim Abduraximov', 'NargizaMirzayeva', 'Raxmonberdiyeva Maxliyo', 'Gulobar Ahmedova', 'Shirmonoy Qurbonova', 'Shodiyona Saydullayeva', 'DilraboQobilova', 'Jalilova Zamira', "Zulayxo Ro'zmaliyeva", "Ulug'bek Isakov", 'Raximova Dilnoza', 'Burxaniddinova Muazzam', 'Erkin

In [28]:
import sys
sys.path.append('/home/hbvision/mirsaid/smart-office')
from ultralytics import YOLO

detector = YOLO("/home/hbvision/mirsaid/smart-office/models/yolov8m-face.pt")

In [33]:
import cv2
import torch
import sys
sys.path.append('/home/hbvision/mirsaid/smart-office')
import numpy as np
from ellzaf_ml.models import GhostFaceNetsV2
from ultralytics import YOLO
import pickle
from sklearn.metrics.pairwise import cosine_similarity

# Initialize constants
IMAGE_SIZE = 256
SIMILARITY_THRESHOLD = 0.6  # Adjust based on your needs

# Load the YOLO detector
detector = YOLO("/home/hbvision/mirsaid/smart-office/models/yolov8m-face.pt")

# Load the face embedding model
model = GhostFaceNetsV2(image_size=IMAGE_SIZE, num_classes=0, width=1, dropout=0.)
model.eval()  # Set to evaluation mode

# Load the database dictionary
try:
    with open("face_embeddings_database.pkl", "rb") as f:
        database_dict = pickle.load(f)
    print(f"Loaded {len(database_dict)} face embeddings from database")
except FileNotFoundError:
    print("Database file not found. Make sure to run the embedding extraction code first.")
    database_dict = {}

# Get database embeddings as numpy arrays for comparison
database_encodings = []
database_names = []
for name, encoding in database_dict.items():
    # If encoding is a tensor, convert to numpy
    if isinstance(encoding, torch.Tensor):
        encoding = encoding.cpu().numpy()
    # If encoding has extra dimensions, flatten it to a 1D array
    if encoding.ndim > 1:
        encoding = encoding.flatten()
    database_encodings.append(encoding)
    database_names.append(name)

# Function to compute face embedding
def get_face_embedding(face_img):
    # Resize to the required dimensions
    face_resized = cv2.resize(face_img, (IMAGE_SIZE, IMAGE_SIZE))
    
    # Convert from BGR to RGB
    face_rgb = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
    
    # Convert to tensor
    face_tensor = torch.from_numpy(face_rgb).float()
    
    # Add batch dimension and rearrange to CHW format
    face_tensor = face_tensor.unsqueeze(0).permute(0, 3, 1, 2)
    
    # Normalize
    face_tensor = face_tensor / 255.0
    
    # Compute embedding
    with torch.no_grad():
        embedding = model(face_tensor)
    
    return embedding.cpu().numpy().flatten()

# Function to find the best match for a face embedding
def identify_face(face_embedding, threshold=SIMILARITY_THRESHOLD):
    if not database_encodings:
        return "Unknown"
    
    # Calculate cosine similarities
    similarities = cosine_similarity([face_embedding], database_encodings)[0]
    
    # Find the best match
    best_match_index = np.argmax(similarities)
    best_similarity = similarities[best_match_index]
    
    # if best_similarity >= threshold:
    return database_names[best_match_index], best_similarity
    # else:
    #     return "Unknown", best_similarity

# Initialize variables
face_locations = []
face_names = []

# Open video capture
video_path = '/home/hbvision/mirsaid/smart-office/client/pred_videos/videoa1-1_eval.mp4'
video_capture = cv2.VideoCapture(video_path)

# Get the dimensions of the input video
width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the codec and create a VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, 20.0, (width, height))

frame_count = 0

while True:
    # Grab a single frame of video
    ret, frame = video_capture.read()
    roi = (302, 82, 986, 976)
    frame = frame[roi[1]:roi[3], roi[0]:roi[2]]

    if not ret:
        print("No frame captured, exiting...")
        break
    
    frame_count += 1
    
    # Run YOLO face detection
    detections = detector(frame, conf=0.5, verbose=False)
    
    if len(detections[0].boxes.data) == 0:
        # If no faces detected, write the original frame and continue
        out.write(frame)
        cv2.imshow('Video', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue
    
    # Get bounding boxes
    boxes = detections[0].boxes.data.cpu().tolist()
    face_locations = []
    face_images = []
    
    for box in boxes:
        # Extract coordinates
        x1, y1, x2, y2 = map(int, box[:4])  # First 4 elements are coordinates
        
        # Store face location as (top, right, bottom, left) - match format from example
        face_locations.append((y1, x2, y2, x1))
        
        # Extract face image
        face_img = frame[y1:y2, x1:x2]
        if face_img.size > 0:  # Check if face image is valid
            face_images.append(face_img)
    
    # Compute embeddings and identify faces
    face_names = []
    for face_img in face_images:
        # Get face embedding
        face_embedding = get_face_embedding(face_img)
        
        # Identify the face
        name, similarity = identify_face(face_embedding)
        
        if name != "Unknown":
            print(f"Detected {name} with similarity {similarity:.2f}")
        
        # Add the name to the list
        face_names.append(f"{name} ({similarity:.2f})")
    
    # Display the results
    for (top, right, bottom, left), name in zip(face_locations, face_names):
        # Draw a box around the face
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 2)
        
        # Draw a label with a name below the face
        cv2.rectangle(frame, (left, bottom - 20), (right, bottom), (0, 0, 255), cv2.FILLED)
        font = cv2.FONT_HERSHEY_DUPLEX
        cv2.putText(frame, name, (left, bottom), font, 0.7, (255, 255, 255), 1)
    
    # Write the frame with face recognition annotations to the output video
    out.write(frame)
    
    # Display the resulting image
    cv2.imshow('Video', frame)
    
    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture and writer, and close all windows
video_capture.release()
out.release()
cv2.destroyAllWindows()

print(f"Processed {frame_count} frames. Output saved to output.avi")

Loaded 99 face embeddings from database
Detected ZuhraUlkanova with similarity 0.06
Detected ZuhraUlkanova with similarity 0.07
Detected ZuhraUlkanova with similarity 0.06
Detected ZuhraUlkanova with similarity 0.07
Detected Dilrabo Egamova with similarity 0.05
Detected ZuhraUlkanova with similarity 0.11
Detected Fayzulloh Saydullayev with similarity 0.06
Detected ZuhraUlkanova with similarity 0.10
Detected Oydina Sharipova with similarity 0.12
Detected Gulmira Rustamova with similarity 0.03
Detected Yulduz Ikramova with similarity 0.04
Detected Raxmonberdiyeva Maxliyo with similarity 0.12
Detected Nasiba Raximova with similarity 0.05
Detected Dilrabo Egamova with similarity 0.06
Detected Ug'iloyHusniddinova with similarity 0.09
Detected ZuhraUlkanova with similarity 0.08
Detected Dilrabo Egamova with similarity 0.09
Detected Shirmonoy Qurbonova with similarity 0.06
Detected Raxmonberdiyeva Maxliyo with similarity 0.06
Detected ZuhraUlkanova with similarity 0.07
Detected Dilrabo Egamov

In [36]:
# SVM Training with FaceNet embeddings
import os
import numpy as np
import cv2
import torch
from facenet_pytorch import InceptionResnetV1
from sklearn import svm
from sklearn.preprocessing import normalize
import joblib
import pickle
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("facenet_svm")

# Configuration
class Args:
    def __init__(self):
        self.db_path = "/home/hbvision/mirsaid/smart-office/data/ilhan2"  # Update this path
        self.device = "cuda:1"
        self.layer = None  # FaceNet layer to use
        self.logger = logger
        self.match_threshold = 0.6
        self.svm_model_path = "facenet_svm_model.joblib"
        self.svm_labels_path = "facenet_svm_labels.pkl"

args = Args()

In [38]:

# Initialize FaceNet model
logger.info(f"Initializing FaceNet model on {args.device}...")
resnet = InceptionResnetV1(pretrained="vggface2", classify=False, layer=args.layer).eval().to(args.device)

# Compute embeddings using FaceNet
def compute_embeddings(faces, model):
    resized_faces = [cv2.resize(face, (160, 160)) for face in faces]
    rgb_faces = [cv2.cvtColor(face, cv2.COLOR_BGR2RGB) for face in resized_faces]
    face_tensors = torch.tensor(np.array(rgb_faces)).permute(0, 3, 1, 2).float().to(args.device) / 255.0
    
    with torch.no_grad():
        embeddings = model(face_tensors).cpu().numpy()
    
    # L2 normalize the embeddings
    normalized_embeddings = normalize(embeddings, norm='l2', axis=1)
    return normalized_embeddings

# Parse person's name from filename
def get_person_name(filename):
    # Extract name before the underscore
    return filename.split('_')[0]

# Load and organize face images by person
logger.info("Loading and organizing face images...")
person_images = defaultdict(list)
person_names = set()

# Load images and group by person
for file in os.listdir(args.db_path):
    if file.lower().endswith((".png", ".jpg", ".jpeg")):
        person_name = get_person_name(file)
        person_names.add(person_name)
        
        face_path = os.path.join(args.db_path, file)
        face_image = cv2.imread(face_path)
        
        if face_image is None:
            logger.warning(f"Could not read image: {face_path}")
            continue
        
        person_images[person_name].append(face_image)

logger.info(f"Found {len(person_names)} unique individuals with {sum(len(imgs) for imgs in person_images.values())} total images")

# Create numeric labels and prepare data for SVM
face_images = []
labels = []
person_id_map = {name: idx for idx, name in enumerate(sorted(person_names))}

for person_name, images in person_images.items():
    person_id = person_id_map[person_name]
    face_images.extend(images)
    labels.extend([person_id] * len(images))

# Compute embeddings in batches to avoid memory issues
logger.info("Computing face embeddings...")
BATCH_SIZE = 32
embeddings = []

for i in range(0, len(face_images), BATCH_SIZE):
    batch_images = face_images[i:i+BATCH_SIZE]
    batch_embeddings = compute_embeddings(batch_images, resnet)
    embeddings.append(batch_embeddings)
    logger.info(f"Processed batch {i//BATCH_SIZE + 1}/{(len(face_images) + BATCH_SIZE - 1)//BATCH_SIZE}")

# Combine all batches
embeddings = np.vstack(embeddings)

INFO:facenet_svm:Initializing FaceNet model on cuda:1...
INFO:facenet_svm:Loading and organizing face images...
INFO:facenet_svm:Found 99 unique individuals with 1287 total images
INFO:facenet_svm:Computing face embeddings...
INFO:facenet_svm:Processed batch 1/41
INFO:facenet_svm:Processed batch 2/41
INFO:facenet_svm:Processed batch 3/41
INFO:facenet_svm:Processed batch 4/41
INFO:facenet_svm:Processed batch 5/41
INFO:facenet_svm:Processed batch 6/41
INFO:facenet_svm:Processed batch 7/41
INFO:facenet_svm:Processed batch 8/41
INFO:facenet_svm:Processed batch 9/41
INFO:facenet_svm:Processed batch 10/41
INFO:facenet_svm:Processed batch 11/41
INFO:facenet_svm:Processed batch 12/41
INFO:facenet_svm:Processed batch 13/41
INFO:facenet_svm:Processed batch 14/41
INFO:facenet_svm:Processed batch 15/41
INFO:facenet_svm:Processed batch 16/41
INFO:facenet_svm:Processed batch 17/41
INFO:facenet_svm:Processed batch 18/41
INFO:facenet_svm:Processed batch 19/41
INFO:facenet_svm:Processed batch 20/41
INF

In [43]:
# Train SVM classifier
logger.info("Training SVM classifier...")
svm_classifier = svm.SVC(C=10, kernel='rbf', probability=True, gamma='scale')
svm_classifier.fit(embeddings, labels)

# Create a mapping from label IDs back to person names
id_to_name = {v: k for k, v in person_id_map.items()}

# Save the SVM model and labels
logger.info("Saving SVM model and labels...")
joblib.dump(svm_classifier, args.svm_model_path)
with open(args.svm_labels_path, 'wb') as f:
    pickle.dump(id_to_name, f)

logger.info(f"SVM training complete! Model saved to {args.svm_model_path}")
logger.info(f"Individual statistics:")
for person_name, images in person_images.items():
    logger.info(f"  - {person_name}: {len(images)} images")

INFO:facenet_svm:Training SVM classifier...
INFO:facenet_svm:Saving SVM model and labels...
INFO:facenet_svm:SVM training complete! Model saved to facenet_svm_model.joblib
INFO:facenet_svm:Individual statistics:
INFO:facenet_svm:  - Zahro Uroqova: 13 images
INFO:facenet_svm:  - Akmal Nazarov: 13 images
INFO:facenet_svm:  - Saboxon Karimova: 13 images
INFO:facenet_svm:  - Dilafruz Yo'ldoshbayeva: 13 images
INFO:facenet_svm:  - Setora: 13 images
INFO:facenet_svm:  - Gulmira Rustamova: 13 images
INFO:facenet_svm:  - NozimaShokirova: 13 images
INFO:facenet_svm:  - DilraboQobilova: 13 images
INFO:facenet_svm:  - FeruzaAtamkulova: 13 images
INFO:facenet_svm:  - Mashrab Muydinov: 13 images
INFO:facenet_svm:  - Oydina Sharipova: 13 images
INFO:facenet_svm:  - MushtariyTurdaliyeva: 13 images
INFO:facenet_svm:  - Erkinoy Atajanova: 13 images
INFO:facenet_svm:  - Sayyora Erkinova Tohirjon qizi: 13 images
INFO:facenet_svm:  - Feruza Yusupova: 13 images
INFO:facenet_svm:  - NasibaJo'rabayeva: 13 im